# 04b. Bán giám sát (Semi-supervised Learning)
Thực nghiệm giả lập thiếu nhãn (chỉ giữ lại 5%, 10%, 20%) và kiểm tra phương pháp Self-Training so với mô hình Supervised-only (XGBoost).

In [ ]:
import pandas as pd
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from src.models.supervised import MaintenancePredictor
from src.models.semi_supervised import SemiSupervisedPredictor

df_cleaned = pd.read_csv('../data/processed/ai4i2020_processed.csv')

## 1. Chuẩn bị dữ liệu ban đầu

In [ ]:
predictor = MaintenancePredictor(config_path='../configs/params.yaml')
X_train_res, X_test_scl, y_train_res, y_test_cls = predictor.split_and_prepare_data(df_cleaned)

## 2. Kịch bản thiếu nhãn 20% và Pseudo-labeling (Self-Training)

In [ ]:
semi = SemiSupervisedPredictor(base_model="xgboost", threshold=0.85)
# Chỉ giữ lại 20% nhãn. Phần còn lại (80%) bị ẩn (-1)
y_semi, y_true_hidden, unlabel_idx = semi.simulate_unlabeled_data(X_train_res, y_train_res, labeled_ratio=0.2)

model_semi = semi.train_and_compare(X_train_res, y_semi, X_test_scl, y_test_cls)

## 3. Phân tích rủi ro Pseudo-label (False Alarm & Missed Failure)

In [ ]:
semi.analyze_pseudo_label_risk(unlabel_idx, y_true_hidden)